<a href="https://colab.research.google.com/github/BenFreilichUVA/walmart-hurricane-project/blob/main/notebooks/team_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

First, we need to clone the repo so we can actually pull the data and work on the project. Then, we need to load the raw data and put it into pandas DataFrames so the data can be manipulated and changed around. There are five raw files in the data folder that need to be converted.

'walmart_transactions.csv', turned into 'transactions'

'store_locations.csv', turned into 'stores'

'product_catalog.csv', turned into 'products'

'distribution_centers.csv', turned into 'centers'

'inventory_and_sales_db', turned into 'daily_sales' and 'inventory'




In [1]:
!rm -rf /content/walmart-hurricane-project
!git clone https://github.com/BenFreilichUVA/walmart-hurricane-project.git /content/walmart-hurricane-project

from pathlib import Path

def check_repo(root):
    root = Path(root)
    needed = ['CHARTER.md', 'README.md', 'data', 'notebooks/team_notebook.ipynb', 'notebooks/scratch']
    ok = True
    for item in needed:
        found = (root / item).exists()
        print('PASS' if found else 'FAIL', '-', item)
        ok = ok and found
    print()
    print('repo ready:', ok)
    return ok

MY_REPO = '/content/walmart-hurricane-project'
check_repo(MY_REPO)

Cloning into '/content/walmart-hurricane-project'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 51 (delta 16), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (51/51), 970.58 KiB | 5.64 MiB/s, done.
Resolving deltas: 100% (16/16), done.
PASS - CHARTER.md
PASS - README.md
PASS - data
PASS - notebooks/team_notebook.ipynb
PASS - notebooks/scratch

repo ready: True


True

In [2]:
DATA = Path(MY_REPO) / 'data'

import pandas as pd
import sqlite3
transactions = pd.read_csv(DATA / 'walmart_transactions.csv')
stores       = pd.read_csv(DATA / 'store_locations.csv')
products     = pd.read_csv(DATA / 'product_catalog.csv')
centers          = pd.read_csv(DATA / 'distribution_centers.csv')

conn = sqlite3.connect(DATA / 'inventory_and_sales.db')

daily_sales = pd.read_sql('SELECT * FROM daily_sales_summary', conn)
inventory   = pd.read_sql('SELECT * FROM inventory_levels', conn)

conn.close()

print('transactions', transactions.shape)
print('stores      ', stores.shape)
print('products    ', products.shape)
print('centers     ', centers.shape)
print('daily_sales', daily_sales.shape)
print('inventory   ', inventory.shape)

decision_log = {}


transactions (25562, 8)
stores       (14, 8)
products     (45, 7)
centers      (5, 8)
daily_sales (3600, 5)
inventory    (4320, 6)


Next, cleaning has to be done. This will take a lot of different steps, and the assignment specifically asks to create a log of each change made.

In [3]:
# Fixing the Store IDS for the transaction DataFrame
def fix_store_ids(df):
    clean = df.copy()
    before = clean['store_id'].nunique()
    clean['store_id'] = (clean['store_id']
                         .str.strip()
                         .str.upper()
                         .str.replace(r'^FL[-_]?', 'FL-', regex=True))
    after = clean['store_id'].nunique()
    # Assertions to ensure this step properly worked
    assert clean['store_id'].nunique() == 12, 'expected 12 stores'
    assert clean['store_id'].str.fullmatch(r'FL-\d{3}').all(), 'some IDs are not in FL-### format'
    assert len(clean) == len(df), 'cleaning should not drop rows'
    decision_log['transactions: store_id'] = f'Standardized store IDs to FL-###, removing unnecessary spacing and uppercasing ({before} spellings condensed to {after} spellings).'
    return clean

# Fixing the Categories for the transaction DataFrame
category_map = {
    'emergency': 'Emergency', 'emergency supplies': 'Emergency', 'grocery': 'Grocery', 'groceries': 'Grocery',
    'electronics': 'Electronics', 'entertainment': 'Entertainment', 'games & entertainment': 'Entertainment',
    'household': 'Household', 'household essentials': 'Household', 'baby': 'Baby', 'baby products': 'Baby',
    'breakfast': 'Breakfast', 'breakfast foods': 'Breakfast', 'alcohol': 'Alcohol', 'alcoholic beverages': 'Alcohol',
    'snacks': 'Snacks', 'health': 'Health', 'health & wellness': 'Health',
}

def fix_categories(df, db_categories):
    clean = df.copy()
    before = clean['category'].nunique()
    clean['category'] = clean['category'].str.strip().str.lower().map(category_map)
    after = clean['category'].nunique()
    #Assertions to ensure this step properly worked
    assert clean['category'].isna().sum() == 0, 'some categories did not map'
    assert set(clean['category']) == set(db_categories), 'categories do not match the database'
    decision_log['transactions: category'] = f'Mapped category names down to the 10 used in the database ({before} spellings condensed to {after} spellings).'
    return clean

# Fixing the unit_price column to not be stored as text, and to instead be stored as a number
def fix_unit_price(df):
    clean = df.copy()
    dollar_rows = clean['unit_price'].astype(str).str.contains('$', regex=False).sum()

    clean['unit_price'] = pd.to_numeric(
        clean['unit_price'].astype(str).str.replace('$', '', regex=False).str.strip(),
        errors='coerce')

    #Assertions to ensure this step properly worked
    assert clean['unit_price'].isna().sum() == 0, 'some prices did not convert'
    assert (clean['unit_price'] > 0).all(), 'found a zero or negative price'
    decision_log['transactions: unit_price'] = f'Removed $ from {dollar_rows:,} prices and converted the column from text to numbers'
    return clean

# Converting all of the timestamps into the same format
formats = ['%Y-%m-%d %H:%M:%S',   # 2024-09-15 18:36:16
           '%Y-%m-%dT%H:%M:%S',   # 2024-09-25T10:00:08
           '%m/%d/%Y %H:%M',      # 09/28/2024 17:19
           '%m-%d-%Y %H:%M:%S',   # 09-26-2024 17:46:57
           '%d-%b-%Y %H:%M']      # 13-Sep-2024 21:07

def parse_timestamps(df):
    clean = df.copy()
    raw_ts = clean['timestamp'].str.strip()
    missing_ts = raw_ts.isna().sum()

    parsed = pd.Series(pd.NaT, index=raw_ts.index, dtype='datetime64[ns]')
    for f in formats:
        attempt = pd.to_datetime(raw_ts, format=f, errors='coerce')
        parsed = parsed.fillna(attempt)

    clean['timestamp'] = parsed

    unparsed = (clean['timestamp'].isna() & raw_ts.notna()).sum()
    # Assertions to ensure the timestamps are correctly formatted.
    assert unparsed == 0, 'some timestamps did not match any format'
    assert clean['timestamp'].isna().sum() == missing_ts, 'blank count changed'
    assert (clean['timestamp'].dropna().dt.month == 9).all(), 'found a date outside September'

    decision_log['transactions: timestamp'] = (f'Parsed 5 timestamp formats into one datetime column '
                                               f'({missing_ts} rows had no timestamp, handled after duplicates were removed)')
    return clean

# Deleting the duplicates
def drop_duplicate_rows(df):
    rows_before = len(df)
    dup_rows = df.duplicated().sum()

    clean = df.drop_duplicates()

    # Assertions to ensure the duplicates were correctly dropped
    assert len(clean) == rows_before - dup_rows, 'dropped the wrong number of rows'
    assert clean['transaction_id'].is_unique, 'some transaction IDs still repeat'

    decision_log['transactions: duplicates'] = (f'Dropped {dup_rows} exact duplicate rows '
                                                f'({rows_before:,} rows became {len(clean):,})')
    return clean

# Fixing the quantities
def fix_quantities(df):
    rows_before = len(df)
    zero_rows = (df['quantity'] == 0).sum()
    neg_rows = (df['quantity'] < 0).sum()
    # Right now we are keeping the negative rows, assuming that they are returns.
    clean = df[df['quantity'] != 0].copy()
    # Assertions to check that there are no zero quantities left in the dataframe.
    assert (clean['quantity'] != 0).all(), 'zero quantities remain'
    assert len(clean) == rows_before - zero_rows, 'dropped the wrong number of rows'

    clean['is_return'] = clean['quantity'] < 0

    decision_log['transactions: quantity'] = (f'Dropped {zero_rows} rows with quantity 0. '
                                              f'Kept {neg_rows} negative quantities as returns, flagged in is_return.')
    return clean

# Creating a second table to only contain orders that have a timestamp
def make_clean_dated(clean):
    blank = clean['timestamp'].isna()
    blank_rows = blank.sum()

    clean_dated = clean[~blank].copy()
    clean_dated['date'] = clean_dated['timestamp'].dt.normalize()
    # Assertions to make sure the new table deleted all of the blank timestamps and that the dates are in the correct range.
    assert clean_dated['timestamp'].isna().sum() == 0, 'blank timestamps remain'
    assert len(clean_dated) == len(clean) - blank_rows, 'clean_dated has the wrong row count'
    assert clean_dated['date'].between('2024-09-01', '2024-09-30').all(), 'date outside September'

    decision_log['transactions: blank timestamps'] = (
        f'Kept {blank_rows} rows with no timestamp in clean ({blank_rows / len(clean):.1%} of rows). '
        f'Left them out of clean_dated, which is used for daily counts (Q1, Q3). ')
    return clean_dated

def clean_transactions(raw, db_categories):
    steps = [fix_store_ids,
             lambda d: fix_categories(d, db_categories),
             fix_unit_price,
             parse_timestamps,
             drop_duplicate_rows,
             fix_quantities]
    names = ['fix_store_ids', 'fix_categories', 'fix_unit_price', 'parse_timestamps', 'drop_duplicate_rows', 'fix_quantities']
    df = raw
    print(f'{"start":20} {len(df):>6,} rows')
    for name, step in zip(names, steps):
        df = step(df)
        print(f'{name:20} {len(df):>6,} rows')
    return df

clean = clean_transactions(transactions, daily_sales['category'])
clean_dated = make_clean_dated(clean)


for step, note in decision_log.items():
    print(f'{step} - {note}')


start                25,562 rows
fix_store_ids        25,562 rows
fix_categories       25,562 rows
fix_unit_price       25,562 rows
parse_timestamps     25,562 rows
drop_duplicate_rows  25,532 rows
fix_quantities       25,532 rows
transactions: store_id - Standardized store IDs to FL-###, removing unnecessary spacing and uppercasing (60 spellings condensed to 12 spellings).
transactions: category - Mapped category names down to the 10 used in the database (37 spellings condensed to 10 spellings).
transactions: unit_price - Removed $ from 3,029 prices and converted the column from text to numbers
transactions: timestamp - Parsed 5 timestamp formats into one datetime column (886 rows had no timestamp, handled after duplicates were removed)
transactions: duplicates - Dropped 30 exact duplicate rows (25,562 rows became 25,532)
transactions: quantity - Dropped 0 rows with quantity 0. Kept 130 negative quantities as returns, flagged in is_return.
transactions: blank timestamps - Kept 884 row

Looking through the stores file, we see that there are problems including messy IDS, duplicate rows, and overall completeness errors. These will be fixed in the cell below.

In [4]:
def clean_stores(stores, clean):
    stores_clean = stores.copy()

    # Fixing the store IDs, same rule as transactions
    stores_clean['store_id'] = (stores_clean['store_id']
                                .str.strip()
                                .str.upper()
                                .str.replace(r'^FL[-_]?', 'FL-', regex=True))

    # Dropping stores that appear twice once their IDs match
    rows_before = len(stores_clean)
    dup_stores = stores_clean.duplicated('store_id').sum()
    stores_clean = stores_clean.drop_duplicates('store_id').copy()

    # Standardizing state and region text
    states_before = stores_clean['state'].unique().tolist()
    stores_clean['state'] = 'FL'
    stores_clean['region'] = stores_clean['region'].str.strip()

    # Filling in Tallahassee's missing coordinates (city center, same as the other stores use)
    missing_coords = stores_clean.loc[stores_clean['latitude'].isna(), 'store_id'].tolist()
    stores_clean.loc[stores_clean['store_id'] == 'FL-619', ['latitude', 'longitude']] = [30.4383, -84.2807]

    # Assertions
    assert len(stores_clean) == 12, 'expected 12 stores'
    assert stores_clean['store_id'].is_unique, 'store IDs repeat'
    assert set(stores_clean['store_id']) == set(clean['store_id']), 'store IDs do not match transactions'
    assert stores_clean[['latitude', 'longitude']].notna().all().all(), 'missing coordinates remain'
    assert stores_clean['region'].nunique() == 6, 'expected 6 regions'

    decision_log['stores: store_id'] = f'Standardized store IDs to FL-### and dropped {dup_stores} duplicate stores ({rows_before} rows became {len(stores_clean)}).'
    decision_log['stores: state'] = f'Set every state to FL (was {states_before}).'
    decision_log['stores: region'] = 'Removed a leading space from " Panhandle" so it matches Panhandle (7 regions condensed to 6).'
    decision_log['stores: coordinates'] = f'Filled missing latitude and longitude for {missing_coords} (Tallahassee) with city-center coordinates.'
    return stores_clean

stores_clean = clean_stores(stores, clean)
print('stores', len(stores), '->', len(stores_clean), 'rows')

stores 14 -> 12 rows


Next is the products DataFrame. Because a lot of this cleaning is less straight forward and requires more judgement calls, a lot of the cleaning here will be saved for Question 2. For now, the cleaning will result on purely objective errors.

In [5]:
def clean_products(products, clean):
    products_clean = products.copy()

    # Fixing categories, reusing the earlier map from the transactions cleaning
    cat_before = products_clean['category'].nunique()
    products_clean['category'] = products_clean['category'].str.strip().str.lower().map(category_map)

    # Standardizing the package_size format to make it more easily comparable
    pkg_before = products_clean['package_size'].nunique()
    products_clean['package_size'] = (products_clean['package_size']
                                      .str.strip().str.lower()
                                      .replace('single', '1pk')
                                      .str.replace(r'^(\d+)\s*-?\s*(pk|pack)$', r'\1-pack', regex=True))

    # Counting what we are deliberately leaving blank
    missing_cost = products_clean['unit_cost'].isna().sum()
    missing_sub = products_clean['subcategory'].isna().sum()

    # Assertions to make sure everything objective that needed to be cleaned worked.
    assert products_clean['category'].isna().sum() == 0, 'some categories did not map'
    assert set(products_clean['category']) == set(clean['category']), 'categories do not match transactions'
    assert set(products_clean['sku']) == set(clean['sku']), 'catalog SKUs do not match transactions'
    assert products_clean['sku'].is_unique, 'SKUs repeat in the catalog'
    assert len(products_clean) == len(products), 'cleaning should not drop rows'

    decision_log['products: category'] = f'Mapped categories to the same 10 as transactions ({cat_before} spellings condensed to 10).'
    decision_log['products: package_size'] = (f'Standardized format to "#-pack" ({pkg_before} spellings condensed to '
                                              f'{products_clean["package_size"].nunique()}). Values contradict product names '
                                              f'(water 24PK listed as 1-pack), so not used in analysis.')
    decision_log['products: supplier'] = 'Left as is. Suppliers do not match products (Roku Inc for Pop-Tarts), so not used in analysis.'
    decision_log['products: missing values'] = (f'Left {missing_cost} missing unit_cost and {missing_sub} missing subcategory blank. '
                                                f'Filling them would mean guessing.')
    decision_log['products: duplicate products'] = 'Kept all 45 SKUs. Several SKUs are the same product, which is Q2, so they are not merged here.'
    return products_clean

products_clean = clean_products(products, clean)
print('products', len(products), '->', len(products_clean), 'rows')

products 45 -> 45 rows


Next we have to fix the Centers Dataframe. Surge capacity was listed as text, and missing shipping costs had to be addressed.

In [6]:
def clean_centers(centers):
    centers_clean = centers.copy()

    # Fixing surge_capacity, which is text because Miami's value is '~750'
    approx_dcs = centers_clean.loc[centers_clean['surge_capacity'].astype(str).str.contains('~', regex=False), 'dc_id'].tolist()
    centers_clean['surge_capacity'] = pd.to_numeric(
        centers_clean['surge_capacity'].astype(str).str.replace('~', '', regex=False).str.strip(),
        errors='coerce')

    # Counting the missing shipping cost we are leaving blank
    missing_cost_dcs = centers_clean.loc[centers_clean['cost_per_shipment'].isna(), 'dc_id'].tolist()

    # Assertions
    assert centers_clean['surge_capacity'].isna().sum() == 0, 'some surge capacities did not convert'
    assert (centers_clean['surge_capacity'] >= centers_clean['avg_daily_shipments']).all(), 'surge capacity below normal shipments'
    assert centers_clean['dc_id'].is_unique, 'distribution centers repeat'
    assert len(centers_clean) == len(centers), 'cleaning should not drop rows'

    decision_log['centers: surge_capacity'] = (f'Removed ~ and converted to numbers. {approx_dcs} was listed as approximate (~750), '
                                               f'kept as 750 but treated as an estimate.')
    # For now we have to leave it as blank because we are unsure. However, we could try to fill this with an estimated model using the other datapoints.
    decision_log['centers: cost_per_shipment'] = (f'Left missing cost for {missing_cost_dcs} (Birmingham) blank. '
                                                  f'Filling it would mean guessing.')
    return centers_clean

centers_clean = clean_centers(centers)
print('centers', len(centers), '->', len(centers_clean), 'rows')

centers 5 -> 5 rows


Next we have to clean the sales DataFrame. The date format is incorrect and needs to be matched to the other DataFrames, so that is where we will start. The negative revenue on positive units is currently being treated as an error. Negative revenue on negative units would make sense as a return, however inverse signs does not make sense. Finally, we have half units, which does not make sense. For now we will merely flag them and leave them as they are.

In [7]:
def clean_sales(daily_sales):
    sales_clean = daily_sales.copy()

    # Fixing dates: removing the dashes makes every date YYYYMMDD, then converting to datetime
    nodash_dates = (sales_clean['date'].str.len() == 8).sum()
    sales_clean['date'] = pd.to_datetime(sales_clean['date'].str.replace('-', '', regex=False), format='%Y%m%d')

    # Fixing negative revenue by flipping the sign and flagging which ones were flipped.
    sales_clean['revenue_sign_fixed'] = sales_clean['total_revenue'] < 0
    neg_revenue = sales_clean['revenue_sign_fixed'].sum()
    sales_clean['total_revenue'] = sales_clean['total_revenue'].abs()

    # Flagging half units rather than rounding randomly.
    sales_clean['half_unit'] = sales_clean['total_units'] % 1 != 0
    half_units = sales_clean['half_unit'].sum()

    # Assertions to make sure the date range is okay, there are no duplicates in the sales, and there are no lines with negative total revenue.
    assert sales_clean['date'].between('2024-09-01', '2024-09-30').all(), 'date outside September'
    assert sales_clean['date'].nunique() == 30, 'expected 30 days'
    assert not sales_clean.duplicated(['date', 'store_id', 'category']).any(), 'a store and category repeat on the same day'
    assert len(sales_clean) == 30 * 12 * 10, 'expected one row per day, store, and category'
    assert (sales_clean['total_revenue'] >= 0).all(), 'negative revenue remains'

    decision_log['daily_sales: date'] = f'Converted {nodash_dates} YYYYMMDD dates and the rest to one datetime column.'
    decision_log['daily_sales: total_revenue'] = (f'Flipped the sign on {neg_revenue} negative revenues. Units were positive and '
                                                  f'revenue per unit matched other rows, so these look like sign errors. Flagged in revenue_sign_fixed.')
    decision_log['daily_sales: total_units'] = f'Left {half_units} half-unit values as is (rounding would be a guess). Flagged in half_unit.'
    return sales_clean

sales_clean = clean_sales(daily_sales)
print('daily_sales', len(daily_sales), '->', len(sales_clean), 'rows')

daily_sales 3600 -> 3600 rows


Finally, we have to clean the inventory table. It needs the date format fixed, the SKUs to be flagged if they are not in the product catalog, and the missing orders.

In [8]:
def clean_inventory(inventory, products_clean, stores_clean):
    inventory_clean = inventory.copy()

    # Converting dates from text to datetime
    inventory_clean['date'] = pd.to_datetime(inventory_clean['date'], format='%Y-%m-%d')

    # Flagging SKUs that are not in the product catalog
    inventory_clean['in_catalog'] = inventory_clean['sku'].isin(products_clean['sku'])
    unknown_skus = sorted(inventory_clean.loc[~inventory_clean['in_catalog'], 'sku'].unique())
    unknown_rows = (~inventory_clean['in_catalog']).sum()

    # Counting missing units_on_order, which we are leaving blank
    missing_orders = inventory_clean['units_on_order'].isna().sum()

    # Assertions to ensure the date format and range are correct, and various variables match everything in the other cleaned tables.
    assert inventory_clean['date'].between('2024-09-01', '2024-09-30').all(), 'date outside September'
    assert inventory_clean['date'].nunique() == 30, 'expected 30 days'
    assert set(inventory_clean['store_id']) == set(stores_clean['store_id']), 'store IDs do not match stores'
    assert not inventory_clean.duplicated(['date', 'store_id', 'sku']).any(), 'a SKU repeats in the same store on the same day'
    assert (inventory_clean['units_on_hand'] >= 0).all(), 'negative stock on hand'
    assert len(inventory_clean) == len(inventory), 'cleaning should not drop rows'

    decision_log['inventory: date'] = 'Converted dates from text to datetime.'
    decision_log['inventory: unknown SKUs'] = (f'Kept {unknown_rows} rows for {len(unknown_skus)} SKUs not in the catalog {unknown_skus}. '
                                               f'Flagged in in_catalog so they can be excluded from product-level analysis.')
    decision_log['inventory: units_on_order'] = (f'Left {missing_orders} missing values blank. Not filled with 0, '
                                                 f'because 0 already means nothing on order.')
    return inventory_clean

inventory_clean = clean_inventory(inventory, products_clean, stores_clean)
print('inventory', len(inventory), '->', len(inventory_clean), 'rows')
print()

for step, note in decision_log.items():
    print(f'{step} - {note}')

inventory 4320 -> 4320 rows

transactions: store_id - Standardized store IDs to FL-###, removing unnecessary spacing and uppercasing (60 spellings condensed to 12 spellings).
transactions: category - Mapped category names down to the 10 used in the database (37 spellings condensed to 10 spellings).
transactions: unit_price - Removed $ from 3,029 prices and converted the column from text to numbers
transactions: timestamp - Parsed 5 timestamp formats into one datetime column (886 rows had no timestamp, handled after duplicates were removed)
transactions: duplicates - Dropped 30 exact duplicate rows (25,562 rows became 25,532)
transactions: quantity - Dropped 0 rows with quantity 0. Kept 130 negative quantities as returns, flagged in is_return.
transactions: blank timestamps - Kept 884 rows with no timestamp in clean (3.5% of rows). Left them out of clean_dated, which is used for daily counts (Q1, Q3). 
stores: store_id - Standardized store IDs to FL-### and dropped 2 duplicate stores (1